> **Traçabilité (fusion Codes_finaux) :** notebook présent uniquement dans `Codes prof/ozone_complet_1_DNN_Keras_tuner.ipynb` (pas d'équivalent Colab), copié tel quel.

---
# DNN : `ozone_complet` (avec `Keras` et `KerasTuner`)
---

## Packages

In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import keras
from keras import callbacks, layers

from keras_tuner.tuners import RandomSearch

---
## 1. Données
---

### 1.1. Importation

In [2]:
# En local :
directory = '/Users/vincentlefieux/Dropbox/Docs_ACADEMIQUE/Data/'

# Sur Google collab ou Onyxia (sur un répertoire temporaire) :
# directory = ''

# Sur Google collab (sur le drive) :
# from google.colab import drive
# drive.mount('/content/drive')
# directory = '/content/drive/MyDrive/Data/'

In [3]:
data = pd.read_csv(directory + 'ozone_complet.csv',
                   header    = 0,
                   index_col = 0,
                   sep       = ';',
                   decimal   = ',')

In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1464 entries, 19950401 to 20020930
Data columns (total 23 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   maxO3   1391 non-null   float64
 1   T6      1463 non-null   float64
 2   T9      1463 non-null   float64
 3   T12     1463 non-null   float64
 4   T15     1463 non-null   float64
 5   T18     1463 non-null   float64
 6   Ne6     1462 non-null   float64
 7   Ne9     1462 non-null   float64
 8   Ne12    1462 non-null   float64
 9   Ne15    1462 non-null   float64
 10  Ne18    1459 non-null   float64
 11  Vdir6   1463 non-null   float64
 12  Vvit6   1463 non-null   float64
 13  Vdir9   1463 non-null   float64
 14  Vvit9   1463 non-null   float64
 15  Vdir12  1463 non-null   float64
 16  Vvit12  1463 non-null   float64
 17  Vdir15  1463 non-null   float64
 18  Vvit15  1463 non-null   float64
 19  Vdir18  1463 non-null   float64
 20  Vvit18  1463 non-null   float64
 21  Vx      1463 non-null   float64

In [5]:
data.head()

,maxO3,T6,T9,T12,T15,T18,Ne6,Ne9,Ne12,Ne15,...,Vdir9,Vvit9,Vdir12,Vvit12,Vdir15,Vvit15,Vdir18,Vvit18,Vx,maxO3v
date,,,,,,,,,,,,,,,,,,,,,
19950401,47.6,10.1,11.6,13.3,13.6,12.2,8.0,8.0,8.0,8.0,...,290.0,4.0,300.0,4.0,340.0,4.0,20.0,4.0,-3.4641,62.2
19950402,56.2,9.5,9.4,13.8,17.4,16.3,8.0,8.0,7.0,0.0,...,160.0,2.0,180.0,3.0,110.0,1.0,350.0,2.0,0.0000,47.6
19950403,61.8,3.6,8.0,16.8,21.5,20.2,4.0,5.0,2.0,2.0,...,20.0,2.0,340.0,1.0,170.0,2.0,170.0,3.0,-0.3420,56.2
19950404,50.8,9.5,10.5,11.4,12.2,11.4,8.0,7.0,7.0,7.0,...,10.0,4.0,350.0,3.0,350.0,3.0,350.0,4.0,-0.5209,61.8
19950405,59.8,9.8,10.8,13.8,14.3,13.3,8.0,7.0,8.0,8.0,...,340.0,2.0,280.0,1.0,320.0,3.0,350.0,4.0,-0.9848,50.8


### 1.2. Gestion des données manquantes

In [6]:
missing_percentage = data.isna().mean() * 100

print("MISSING VALUES :")
if missing_percentage[missing_percentage != 0].empty:
    print("No")
else:
    print(missing_percentage[missing_percentage != 0].sort_values(ascending=False))

MISSING VALUES :
maxO3     4.986339
maxO3v    4.986339
Ne18      0.341530
Ne6       0.136612
Ne9       0.136612
Ne12      0.136612
Ne15      0.136612
T9        0.068306
Vdir12    0.068306
Vx        0.068306
Vvit18    0.068306
Vdir18    0.068306
Vvit15    0.068306
Vdir15    0.068306
Vvit12    0.068306
Vvit6     0.068306
Vvit9     0.068306
Vdir9     0.068306
T12       0.068306
T6        0.068306
T18       0.068306
T15       0.068306
Vdir6     0.068306
dtype: float64


In [7]:
data.dropna(inplace=True)

### 1.3. Gestion des variables

In [8]:
target = 'maxO3'

y = data[target]
X = data.drop(target, axis=1)

### 1.4. Création des échantillons de validation et test

In [9]:
test_portion  = 1/5
valid_portion = 1/5

X_train_valid, X_test, y_train_valid, y_test = train_test_split(X, y, test_size=test_portion)

X_train, X_valid, y_train, y_valid = train_test_split(X_train_valid, y_train_valid, test_size=valid_portion)

print('Dimensions de X_train :', X_train.shape)
print('Dimensions de X_valid :', X_valid.shape)
print('Dimensions de X_test  :', X_test.shape)

print('Dimensions de y_train :', y_train.shape)
print('Dimensions de y_valid :', y_valid.shape)
print('Dimensions de y_test  :', y_test.shape)

Dimensions de X_train : (873, 22)
Dimensions de X_valid : (219, 22)
Dimensions de X_test  : (274, 22)
Dimensions de y_train : (873,)
Dimensions de y_valid : (219,)
Dimensions de y_test  : (274,)


### 1.5. Normalisation des covariables

On normalise (centrage-réduction) les covariables :

In [10]:
scaler = StandardScaler()
scaler.fit(X_train)

X_train_norm = scaler.transform(X_train)
X_valid_norm = scaler.transform(X_valid)
X_test_norm  = scaler.transform(X_test)

---
## 2. DNN
---

### 2.1. Architecture et optimiseur

In [11]:
dim_outputs = 1

n_layers_min = 1
n_layers_max = 3

n_units_min  = 32
n_units_max  = 128
n_units_step = 32

dropout_min  = 0.2
dropout_max  = 0.2
dropout_step = 0.1

activation_hidden = 'relu'
activation_output = 'linear'

optimizer = 'adam'
# learning_rate_values = [1e-3, 5e-4, 1e-4, 5e-5, 1e-5]
loss      = 'mse'
metrics   = ['mae']

n_epochs_max = 200
batch_size   = 300
patience     = 10
verbose      = 0

def build_model(hp):
    model = keras.Sequential()
    for i in range(hp.Int('num_layers', n_layers_min, n_layers_max)):
        model.add(layers.Dense(units = hp.Int('units_' + str(i+1),
                                              min_value = n_units_min,
                                              max_value = n_units_max,
                                              step      = n_units_step),
                               activation = activation_hidden,
                               name = 'Hidden_layer_' + str(i+1)))
        model.add(layers.Dropout(rate=hp.Float('dropout_' + str(i+1),
                                                min_value = dropout_min,
                                                max_value = dropout_max,
                                                step      = dropout_step)))
    model.add(layers.Dense(dim_outputs, activation=activation_output, name='Output_layer'))
    
    model.compile(optimizer   = optimizer,
                  # optimizer = keras.optimizers.Adam(hp.Choice('learning_rate', learning_rate_values)),
                  loss        = loss,
                  metrics     = metrics)
    
    return model

### 2.2. Entraînement

In [12]:
objective              = 'val_loss'
n_trials               = 50
n_executions_per_trial = 1
directory              = 'Project'
project_name           = 'Ozone_complet_Tuner'

monitor  = 'val_loss'
mode     = 'min'
patience = 10

batch_size   = 300
n_epochs_max = 200
verbose      = 0

tuner = RandomSearch(build_model,
                     objective            = objective,
                     max_trials           = n_trials,
                     executions_per_trial = n_executions_per_trial,
                     directory            = directory,
                     project_name         = project_name)

callback = callbacks.EarlyStopping(monitor              = monitor,
                                   mode                 = mode,
                                   patience             = patience,
                                   restore_best_weights = True)

tuner.search(X_train_norm,
             y_train,
             batch_size      = batch_size,
             epochs          = n_epochs_max,
             validation_data = (X_valid_norm, y_valid),
             callbacks       = [callback],
             verbose         = verbose)

In [13]:
best_hp = tuner.get_best_hyperparameters()[0]

print(f"Nombre de couches : {best_hp.get('num_layers')}")

for i in range(1, best_hp.get('num_layers') + 1):
    print(f"Nombre de neurones couche {i} : {best_hp.get(f'units_{i}')}")

Nombre de couches : 3
Nombre de neurones couche 1 : 128
Nombre de neurones couche 2 : 64
Nombre de neurones couche 3 : 96


In [14]:
best_model = tuner.get_best_models(num_models=1)[0]

/Users/vincentlefieux/anaconda3/envs/cepe/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:719: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


### 2.3. Prévisions

In [15]:
y_test_pred = best_model.predict(X_test_norm)
y_test_pred[0:5]

9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 


array([[ 89.70824 ],
       [ 91.83391 ],
       [130.63943 ],
       [ 68.312836],
       [ 95.61341 ]], dtype=float32)

In [16]:
score_test = best_model.evaluate(X_test_norm, y_test, verbose=0)
print(f'MSE test : {score_test[0]:4.4f}')
print(f'MAE test : {score_test[1]:4.4f}')

MSE test : 167.1003
MAE test : 9.9574
